# PROJETO 01: Análise e Visualização de Dados
## Análise Exploratória: Tendências e Evolução do Setor de Beleza no Mercado de

*   Item da lista
*   Item da lista

Capitais (2025-2026)
**Professor:** Eronides da Silva Neto
**Curso:** ---
**Dataset:** --
**Alunos:** Ana Paula Ferreira Pessoa

---
### **Visão Geral do Projeto & Contexto de Consultoria**


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker


# Função para processar o layout COTAHIST da B3
def processar_cotahist(caminho_arquivo):
    widths = [2, 8, 2, 12, 3, 12, 10, 3, 4, 13, 13, 13, 13, 13, 13, 13, 5, 18, 18, 13, 1, 8, 7, 13, 12, 3]

    colunas = [
        'TIPREG', 'DATPRG', 'CODBDI', 'CODNEG', 'TPMERC', 'NOMRES', 'ESPECI',
        'PRAZOT', 'MODREF', 'PREABE', 'PREMAX', 'PREMIN', 'PREMED', 'PREULT',
        'PREOFC', 'PREOFV', 'TOTNEG', 'QUATOT', 'VOLTOT', 'PREEXE', 'INDOPC',
        'DATVEN', 'FATCOT', 'PTOEXE', 'CODISI', 'DISMES'
    ]

    df = pd.read_fwf(caminho_arquivo, widths=widths, names=colunas, dtype=str)
    df = df[df['TIPREG'] == '01'].copy()
    df['DATPRG'] = pd.to_datetime(df['DATPRG'], format='%Y%m%d', errors='coerce')

    colunas_financeiras = [
        'PREABE', 'PREMAX', 'PREMIN', 'PREMED', 'PREULT',
        'PREOFC', 'PREOFV', 'VOLTOT', 'PREEXE'
    ]

    for col in colunas_financeiras:
        df[col] = pd.to_numeric(df[col], errors='coerce') / 100.0

    df['TOTNEG'] = pd.to_numeric(df['TOTNEG'], errors='coerce')
    df['QUATOT'] = pd.to_numeric(df['QUATOT'], errors='coerce')
    df = df.drop(columns=['TIPREG'])

    return df

# CAMINHOS CORRIGIDOS COM O ESPAÇO NO FINAL DA PASTA
caminho_pasta = '/content/drive/MyDrive/Atividade 01 - Análise e Visualização de Dados /'
arquivo_2025 = caminho_pasta + 'COTAHIST_A2025.TXT'
arquivo_2026 = caminho_pasta + 'COTAHIST_A2026.TXT'

# Lê e processa ambos os arquivos
print("Lendo o arquivo de 2025...")
df_2025 = processar_cotahist(arquivo_2025)

print("Lendo o arquivo de 2026...")
df_2026 = processar_cotahist(arquivo_2026)

# Concatena os DataFrames em um só
print("Concatenando os dados...")
df_historico = pd.concat([df_2025, df_2026], ignore_index=True)

# ------------------------------------------------------------------
# CORREÇÃO (Etapa 1 - Auditoria, revisada após validação com dados reais)
# ------------------------------------------------------------------
# Primeira tentativa de correção: filtrar CODBDI == '02' (lote padrão),
# para evitar duplicidade de (data, ticker) por causa de mercado
# fracionário. TESTADO COM OS DADOS REAIS e descartado: COTY34 é um
# BDR e usa CODBDI == '34', não '02' — esse filtro estava excluindo a
# COTY34 inteira da análise (todos os seus registros).
#
# A validação com os dados reais mostrou que os 6 tickers deste
# comparativo (incluindo NATU3, ver célula seguinte) NÃO têm registros
# duplicados por (DATPRG, CODNEG) mesmo sem filtrar CODBDI — ou seja, o
# problema teórico de duplicidade (lote padrão x fracionário) não se
# concretiza neste dataset para essas empresas específicas. Por isso,
# mantemos todos os CODBDI aqui e confiamos na verificação de
# duplicidade abaixo como salvaguarda (ela dispara um alerta se algum
# dia isso deixar de ser verdade, por exemplo se novos tickers forem
# adicionados à lista).
print("\nCódigos de CODBDI encontrados no dataset:")
print(df_historico['CODBDI'].value_counts())

# Verificação de duplicidade por (data, ticker) — salvaguarda, não filtro
duplicados = df_historico.duplicated(subset=['DATPRG', 'CODNEG']).sum()
if duplicados > 0:
    print(
        f"\n⚠️ Atenção: {duplicados} registros duplicados de (DATPRG, CODNEG) "
        f"encontrados no dataset completo (esperado para tickers fora deste "
        f"comparativo, ex.: ativos com mercado fracionário). Isso é filtrado "
        f"na célula seguinte, que seleciona apenas os 6 tickers de interesse."
    )
else:
    print("\nNenhuma duplicidade de (DATPRG, CODNEG) encontrada no dataset completo.")

# Exibe os resultados
print("\nDimensões do DataFrame consolidado:", df_historico.shape)
display(df_historico.head())


In [ ]:
# 1. Limpando os espaços em branco da coluna CODNEG
df_historico['CODNEG'] = df_historico['CODNEG'].str.strip()

# 2. Definindo as ações do setor de beleza e cuidados pessoais
empresas_beleza = [
    'NTCO3',
    'COTY34',
    'HYPE3',
    'PNVL3',
    'RADL3',
    'ESPA3'
]

# ------------------------------------------------------------------
# NOTA IMPORTANTE — Virada de ticker NTCO3 → NATU3 (01/07/2025)
# ------------------------------------------------------------------
# Em 01/07/2025, a Natura&Co Holding (ticker NTCO3) foi incorporada
# pela Natura Cosméticos. A partir de 02/07/2025, as ações passaram a
# ser negociadas sob o código NATU3 — mesma empresa, conversão de
# 1 ação NTCO3 para 1 ação NATU3, sem ajuste de preço. Sem tratar
# isso, a série da "NTCO3" simplesmente pararia em 01/07/2025 no
# dataset, enquanto as demais empresas seguem até 2026 — o que
# inviabilizaria qualquer comparação justa ao longo de todo o período.
#
# Por isso, tratamos NATU3 como a mesma empresa que NTCO3: os
# registros de NATU3 são incluídos no filtro abaixo e depois
# relabelados de CODNEG='NATU3' para CODNEG='NTCO3', formando uma
# única série contínua.
codigos_para_filtrar = empresas_beleza + ['NATU3']

# 3. Paleta de cores das marcas
# Cada empresa possui 3 variações:
# Escuro = #1
# Principal = #2
# Claro = #3

cores_marcas = {
    'NTCO3': {
        'escuro': '#B8750F',
        'principal': '#F4AB34',
        'claro': '#F9D28A'
    },
    'RADL3': {
        'escuro': '#006B3C',
        'principal': '#00A859',
        'claro': '#66D1A0'
    },
    'HYPE3': {
        'escuro': '#0755A0',
        'principal': '#1185F0',
        'claro': '#70B7F7'
    },
    'PNVL3': {
        'escuro': '#01245A',
        'principal': '#013684',
        'claro': '#6685B0'
    },
    'ESPA3': {
        'escuro': '#007A9D',
        'principal': '#00B2E4',
        'claro': '#66D0EE'
    },
    'COTY34': {
        'escuro': '#3E1F43',
        'principal': '#63316B',
        'claro': '#A984AE'
    }
}

# 4. Definindo o estilo global do Seaborn
sns.set_theme(style="whitegrid")

# 5. Filtrando o DataFrame para incluir as 6 empresas + NATU3 (temporariamente)
df_beleza = df_historico[
    df_historico['CODNEG'].isin(codigos_para_filtrar)
].copy()

# Relabela NATU3 como NTCO3 (mesma empresa, série contínua — ver nota acima)
_registros_natu3 = int((df_beleza['CODNEG'] == 'NATU3').sum())
df_beleza.loc[df_beleza['CODNEG'] == 'NATU3', 'CODNEG'] = 'NTCO3'
if _registros_natu3 > 0:
    print(
        f"{_registros_natu3} registros de NATU3 foram unidos à série da "
        f"NTCO3 (mesma empresa; ticker renomeado em 01/07/2025)."
    )

# Verificação: como NTCO3 encerrou em 01/07/2025 e NATU3 começou em
# 02/07/2025, não deveria haver datas sobrepostas na série unificada.
_duplicadas_ntco3 = df_beleza[df_beleza['CODNEG'] == 'NTCO3'].duplicated(subset=['DATPRG']).sum()
if _duplicadas_ntco3 > 0:
    print(f"⚠️ Atenção: {_duplicadas_ntco3} datas duplicadas na série unificada da NTCO3.")

# ------------------------------------------------------------------
# NOTA IMPORTANTE — Grupamento de ações da ESPA3, 10 para 1 (15/06/2026)
# ------------------------------------------------------------------
# Em 15/06/2026, a Espaçolaser (ESPA3) fez um grupamento de ações na
# proporção de 10 para 1 (cada 10 ações antigas viraram 1 ação nova),
# para sair da faixa de "penny stock" exigida pela B3. Isso NÃO é
# crescimento real de preço: sem ajuste, a série mostraria um "salto"
# de ~10x de uma sessão pra outra (confirmado nos dados: R$ 0,73 em
# 12/06/2026 para R$ 7,08 em 15/06/2026), distorcendo qualquer cálculo
# de retorno, volatilidade ou drawdown da ESPA3.
#
# Ajuste: preços ANTES do grupamento são multiplicados por 10, e a
# quantidade negociada (QUATOT) é dividida por 10, para deixar a série
# inteira em base "pós-grupamento" (comparável a preço atual). O
# volume financeiro (VOLTOT) não precisa de ajuste — o valor em R$
# negociado não muda com o grupamento.
DATA_GRUPAMENTO_ESPA3 = pd.Timestamp('2026-06-15')
_mascara_pre_grupamento = (df_beleza['CODNEG'] == 'ESPA3') & (df_beleza['DATPRG'] < DATA_GRUPAMENTO_ESPA3)
_registros_ajustados = int(_mascara_pre_grupamento.sum())

_colunas_preco = ['PREABE', 'PREMAX', 'PREMIN', 'PREMED', 'PREULT', 'PREOFC', 'PREOFV']
for _col in _colunas_preco:
    df_beleza.loc[_mascara_pre_grupamento, _col] *= 10

df_beleza.loc[_mascara_pre_grupamento, 'QUATOT'] /= 10

if _registros_ajustados > 0:
    print(
        f"{_registros_ajustados} registros de ESPA3 anteriores ao "
        f"grupamento (15/06/2026) foram ajustados (preços x10, "
        f"quantidade /10) para manter a série comparável."
    )

# 6. Exibindo o resultado
print(
    f"Total de registros encontrados para o setor de beleza: "
    f"{len(df_beleza)}"
)

display(df_beleza.head())

# 7. Quantidade de registros por empresa
print("\nRegistros por empresa:")
print(df_beleza['CODNEG'].value_counts())


In [ ]:
# ==========================================
# ETAPA 2 — IDENTIDADE VISUAL PADRONIZADA
# ==========================================
# Centraliza aqui as regras de cor e formatação que serão reutilizadas
# em todos os gráficos da Etapa 3 e no infográfico da Etapa 4.
# Nenhum gráfico existente é alterado nesta célula — apenas os
# "insumos" visuais (paleta, formatação, estilo) ficam definidos.

EMPRESA_FOCO = 'NTCO3'


def cor_empresa(empresa, contexto='comparativo'):
    """
    Retorna a cor de uma empresa seguindo a regra de destaque:
    - NTCO3 (empresa foco) sempre usa a cor de destaque (tier 'escuro'),
      mais saturada, para ser imediatamente identificável nos gráficos
      comparativos.
    - As demais empresas usam uma cor secundária (tier 'claro'), mais
      discreta, para não competir visualmente com a NTCO3.

    contexto='unico' força o uso da cor 'principal' (tier intermediário),
    útil em gráficos onde apenas uma empresa aparece por vez.
    """
    if contexto == 'unico':
        return cores_marcas[empresa]['principal']
    if empresa == EMPRESA_FOCO:
        return cores_marcas[empresa]['escuro']
    return cores_marcas[empresa]['claro']


# Paleta pronta para uso em gráficos comparativos (NTCO3 em destaque,
# demais em tom secundário)
paleta_comparativa = {empresa: cor_empresa(empresa) for empresa in empresas_beleza}

# ------------------------------------------------------------------
# Padrões de formatação (aplicados nos gráficos a partir da Etapa 3)
# ------------------------------------------------------------------
CASAS_DECIMAIS_PRECO = 2
CASAS_DECIMAIS_PERCENTUAL = 2


def formatar_moeda(valor):
    """Formata valor em Real, padrão brasileiro (ex.: R$ 1.234,56)."""
    texto = 'R$ {:,.2f}'.format(valor)
    return texto.replace(',', 'X').replace('.', ',').replace('X', '.')


def formatar_percentual(valor):
    """Formata valor em percentual com vírgula decimal (ex.: 12,34%)."""
    return '{:.2f}%'.format(valor).replace('.', ',')


# ------------------------------------------------------------------
# Estilo global dos gráficos: fonte, tamanhos e grid discreto
# ------------------------------------------------------------------
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COR_TEXTO_SECUNDARIO = '#667085'
COR_NEUTRO_REFERENCIA = '#98A2B3'

print(f"Empresa foco: {EMPRESA_FOCO}")
print("Paleta comparativa (NTCO3 em destaque, demais em tom secundário):")
for empresa, cor in paleta_comparativa.items():
    destaque = " ← DESTAQUE" if empresa == EMPRESA_FOCO else ""
    print(f"  {empresa}: {cor}{destaque}")


In [35]:
# Tabela de KPIs
kpis = []

for empresa in empresas_beleza:
    df_empresa = df_beleza[df_beleza['CODNEG'] == empresa].sort_values('DATPRG')

    if not df_empresa.empty:
        preco_inicial = df_empresa['PREULT'].iloc[0]
        preco_final = df_empresa['PREULT'].iloc[-1]
        crescimento_pct = ((preco_final - preco_inicial) / preco_inicial) * 100

        kpis.append({
            'Ativo': empresa,
            'Preço Inicial (2025)': preco_inicial,
            'Preço Final (2026)': preco_final,
            'Mínimo no Período': df_empresa['PREULT'].min(),
            'Máximo no Período': df_empresa['PREULT'].max(),
            'Crescimento Acumulado (%)': round(crescimento_pct, 2)
        })

df_kpis = pd.DataFrame(kpis).set_index('Ativo')
display(df_kpis.style.background_gradient(cmap='RdYlGn', subset=['Crescimento Acumulado (%)']))

,Preço Inicial (2025),Preço Final (2026),Mínimo no Período,Máximo no Período,Crescimento Acumulado (%)
Ativo,,,,,
NTCO3,12.470000,10.800000,8.980000,13.990000,-13.390000
COTY34,21.380000,7.210000,4.710000,21.840000,-66.280000
HYPE3,17.830000,20.730000,17.820000,28.410000,16.260000
PNVL3,8.600000,11.090000,7.890000,16.340000,28.950000
RADL3,21.760000,17.570000,13.360000,27.000000,-19.260000
ESPA3,0.730000,6.800000,0.670000,8.490000,831.510000


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ------------------------------------------------------------------
# ETAPA 3 — Padronização visual: Boxplot de dispersão de preços
# ------------------------------------------------------------------
sns.boxplot(
    data=df_beleza, x='CODNEG', y='PREULT', hue='CODNEG', legend=False,
    palette=paleta_comparativa, order=empresas_beleza, hue_order=empresas_beleza, ax=axes[0]
)

# Título orientado a insight: identifica, a partir dos dados carregados,
# qual empresa tem a maior dispersão de preços (IQR) — sem presumir
# de antemão qual seria o resultado.
iqr_por_empresa = df_beleza.groupby('CODNEG')['PREULT'].apply(
    lambda s: s.quantile(0.75) - s.quantile(0.25)
)
empresa_maior_iqr = iqr_por_empresa.idxmax()
if empresa_maior_iqr == EMPRESA_FOCO:
    titulo_boxplot = f"{EMPRESA_FOCO} apresentou a maior dispersão de preços do grupo"
else:
    titulo_boxplot = f"{empresa_maior_iqr} apresentou maior dispersão de preços que {EMPRESA_FOCO}"

axes[0].set_title(titulo_boxplot)
axes[0].set_ylabel("Preço de Fechamento (R$)")
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda valor, pos: formatar_moeda(valor)))

# ------------------------------------------------------------------
# ETAPA 3 — Padronização visual: Histograma e densidade dos retornos
# ------------------------------------------------------------------
df_pivot = df_beleza.pivot_table(index='DATPRG', columns='CODNEG', values='PREULT')
df_retornos = df_pivot.pct_change() * 100  # Multiplica por 100 para %

for empresa in empresas_beleza:
    if empresa in df_retornos.columns:
        eh_foco = empresa == EMPRESA_FOCO
        sns.histplot(
            df_retornos[empresa].dropna(), kde=True, stat="density",
            color=cor_empresa(empresa), label=empresa,
            alpha=0.55 if eh_foco else 0.25,
            linewidth=1.6 if eh_foco else 0.8,
            ax=axes[1]
        )

# Título orientado a insight: compara a volatilidade (desvio padrão dos
# retornos diários) da NTCO3 com a média das comparáveis.
vol_por_empresa = df_retornos[empresas_beleza].std()
vol_ntco3 = vol_por_empresa[EMPRESA_FOCO]
vol_media_comparaveis = vol_por_empresa.drop(EMPRESA_FOCO).mean()
if vol_ntco3 > vol_media_comparaveis:
    titulo_hist = f"{EMPRESA_FOCO} apresentou volatilidade diária acima da média do grupo comparável"
else:
    titulo_hist = f"{EMPRESA_FOCO} apresentou volatilidade diária abaixo da média do grupo comparável"

axes[1].set_title(titulo_hist)
axes[1].set_xlabel("Variação Diária (%)")
axes[1].set_ylabel("Densidade")
axes[1].legend(frameon=False, ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# ETAPA 3 — Padronização visual: Heatmap de correlação
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 6))

correlacao = df_retornos.corr()

sns.heatmap(
    correlacao, annot=True, cmap='coolwarm', vmin=-1, vmax=1,
    linewidths=0.5, fmt=".2f", cbar_kws={'label': 'Correlação'}, ax=ax
)

# Destaque visual: contorna a linha e a coluna da empresa foco (NTCO3)
posicao_ntco3 = list(correlacao.columns).index(EMPRESA_FOCO)
ax.add_patch(
    patches.Rectangle(
        (posicao_ntco3, 0), 1, len(correlacao),
        fill=False, edgecolor=cores_marcas[EMPRESA_FOCO]['escuro'], linewidth=2.5
    )
)
ax.add_patch(
    patches.Rectangle(
        (0, posicao_ntco3), len(correlacao), 1,
        fill=False, edgecolor=cores_marcas[EMPRESA_FOCO]['escuro'], linewidth=2.5
    )
)

# Título orientado a insight: identifica a empresa com comportamento
# mais e menos parecido com o da NTCO3 (maior/menor correlação de retornos).
corr_com_ntco3 = correlacao[EMPRESA_FOCO].drop(EMPRESA_FOCO)
empresa_mais_correlacionada = corr_com_ntco3.idxmax()
ax.set_title(
    f"{empresa_mais_correlacionada} é a empresa com retornos mais parecidos aos da {EMPRESA_FOCO}"
)

plt.tight_layout()
plt.show()


In [ ]:
# Criando coluna de Ano-Mês para agrupamento temporal
df_beleza['AnoMes'] = df_beleza['DATPRG'].dt.to_period('M')

# Agrupando volume total por mês e empresa
df_volume_mes = df_beleza.groupby(['AnoMes', 'CODNEG'])['VOLTOT'].sum().unstack().fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={'width_ratios': [2, 1]})

# ------------------------------------------------------------------
# ETAPA 3 — Padronização visual: Volume mensal negociado (stacked bars)
# ------------------------------------------------------------------
cores_barras = [cor_empresa(empresa) for empresa in df_volume_mes.columns]
df_volume_mes.plot(
    kind='bar', stacked=True, color=cores_barras,
    ax=axes[0], edgecolor='white'
)

# Título orientado a insight: tendência do volume da NTCO3 no período
# (comparando o primeiro e o último mês da série).
tendencia_ntco3 = (
    "crescente" if df_volume_mes[EMPRESA_FOCO].iloc[-1] > df_volume_mes[EMPRESA_FOCO].iloc[0]
    else "decrescente"
)
axes[0].set_title(f"Volume mensal negociado — {EMPRESA_FOCO} com tendência {tendencia_ntco3} no período")
axes[0].set_ylabel("Volume Total (R$)")
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda valor, pos: formatar_moeda(valor)))
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(frameon=False, ncol=3, title=None)

# ------------------------------------------------------------------
# ETAPA 3 — Padronização visual: Share de liquidez no período (pizza)
# ------------------------------------------------------------------
volume_total_empresa = df_beleza.groupby('CODNEG')['VOLTOT'].sum()
cores_pizza = [cor_empresa(empresa) for empresa in volume_total_empresa.index]
axes[1].pie(
    volume_total_empresa, labels=volume_total_empresa.index,
    colors=cores_pizza,
    autopct=lambda pct: formatar_percentual(pct),
    startangle=90, wedgeprops={'edgecolor': 'white'}
)

# Título orientado a insight: participação real da NTCO3 no volume total.
share_ntco3 = volume_total_empresa[EMPRESA_FOCO] / volume_total_empresa.sum() * 100
axes[1].set_title(f"{EMPRESA_FOCO} responde por {formatar_percentual(share_ntco3)} do volume negociado no grupo")

plt.tight_layout()
plt.show()


In [ ]:
from scipy import stats

# ------------------------------------------------------------------
# CORREÇÃO (Etapa 1 - Auditoria): cálculo redundante removido
# ------------------------------------------------------------------
# `df_pivot`/`df_retornos` já foram calculados na célula do
# boxplot/histograma (mais acima) e não são alterados depois disso —
# reutilizamos a mesma variável em vez de recalcular do zero.

# 1. Definindo o Teste A/B entre duas marcas de interesse (Grupo A: NTCO3 vs Grupo B: RADL3)
grupo_a = 'NTCO3'
grupo_b = 'RADL3'

# Removendo valores nulos para a comparação (cada amostra usa apenas
# suas próprias observações válidas — não é necessário alinhar datas
# para um teste t de amostras independentes)
dados_a = df_retornos[grupo_a].dropna()
dados_b = df_retornos[grupo_b].dropna()

# 2. Executando o Teste T de Student (Teste A/B estatístico)
# equal_var=False (Welch) é a escolha correta aqui, pois não há
# garantia de que as variâncias dos retornos das duas empresas sejam iguais.
t_stat, p_value = stats.ttest_ind(dados_a, dados_b, equal_var=False)

# 3. Exibindo os resultados do Teste A/B
print(f"=== TESTE A/B: Retorno Diário ({grupo_a} vs {grupo_b}) ===")
print(f"Média diária - {grupo_a}: {dados_a.mean():.4f}%")
print(f"Média diária - {grupo_b}: {dados_b.mean():.4f}%")
print(f"Estatística t: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Interpretando o resultado estatístico (considerando nível de significância de 5% ou 0.05)
alpha = 0.05
if p_value < alpha:
    print("\nResultado: Rejeitamos a hipótese nula. Há uma diferença estatisticamente significativa entre os retornos diários das duas empresas.")
else:
    print("\nResultado: Não há evidências estatísticas suficientes para afirmar que os retornos diários médios são diferentes entre as duas empresas.")


## Etapa 5 — Novos gráficos comparativos (NTCO3 vs. grupo)

Cada gráfico abaixo responde a uma pergunta analítica específica e alimenta um bloco do infográfico executivo (Blocos 2, 3 e 4).

Observação de escopo: "retorno acumulado" e "NTCO3 vs. mediana do grupo" foram consolidados em um único gráfico (evolução indexada base 100), já que são matematicamente a mesma série temporal — mostrá-los separadamente seria redundante.

In [ ]:
# ------------------------------------------------------------------
# ETAPA 5.1 — Evolução temporal indexada (base 100) vs. mediana do grupo
# ------------------------------------------------------------------
# Pergunta que este gráfico responde: ao longo do tempo, a NTCO3 supera
# ou fica abaixo do "concorrente típico" (mediana das comparáveis)?
# Usamos índice base 100 para comparar empresas com preços absolutos
# muito diferentes, e mediana (não média) como referência do grupo
# comparável, por ser menos sensível a valores extremos com apenas
# 5 comparáveis.

fig, ax = plt.subplots(figsize=(14, 6))

primeiro_valor_valido = df_pivot.apply(lambda coluna: coluna.dropna().iloc[0])
df_indexado = df_pivot.divide(primeiro_valor_valido, axis=1) * 100

mediana_grupo_comparavel = df_indexado.drop(columns=EMPRESA_FOCO).median(axis=1)

for empresa in empresas_beleza:
    if empresa == EMPRESA_FOCO:
        continue
    ax.plot(
        df_indexado.index, df_indexado[empresa],
        color=cor_empresa(empresa), linewidth=1.1, alpha=0.55, zorder=2
    )

ax.plot(
    df_indexado.index, mediana_grupo_comparavel,
    color=COR_TEXTO_SECUNDARIO, linewidth=1.6, linestyle='--',
    label='Mediana do grupo comparável', zorder=3
)
ax.plot(
    df_indexado.index, df_indexado[EMPRESA_FOCO],
    color=cor_empresa(EMPRESA_FOCO), linewidth=2.8,
    label=EMPRESA_FOCO, zorder=4
)
ax.axhline(100, color=COR_NEUTRO_REFERENCIA, linewidth=0.8, alpha=0.6)

# Título orientado a insight: NTCO3 acima ou abaixo da mediana no final do período
variacao_ntco3 = df_indexado[EMPRESA_FOCO].iloc[-1] - 100
variacao_mediana_grupo = mediana_grupo_comparavel.iloc[-1] - 100
if df_indexado[EMPRESA_FOCO].iloc[-1] > mediana_grupo_comparavel.iloc[-1]:
    titulo_indexado = f"{EMPRESA_FOCO} supera a mediana do grupo comparável no acumulado do período"
else:
    titulo_indexado = f"{EMPRESA_FOCO} fica abaixo da mediana do grupo comparável no acumulado do período"

ax.set_title(titulo_indexado)
ax.set_ylabel("Índice (base 100 no início do período)")
ax.set_xlabel("")
ax.legend(frameon=False, ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# ETAPA 5.2 — Dispersão risco × retorno
# ------------------------------------------------------------------
# Pergunta que este gráfico responde: a NTCO3 tem uma relação
# risco-retorno favorável frente às comparáveis? Eixo X = risco
# (volatilidade diária), eixo Y = retorno médio diário, cada ponto
# é uma empresa.

fig, ax = plt.subplots(figsize=(9, 7))

retorno_medio_diario = df_retornos[empresas_beleza].mean()
volatilidade_diaria = df_retornos[empresas_beleza].std()

for empresa in empresas_beleza:
    eh_foco = empresa == EMPRESA_FOCO
    ax.scatter(
        volatilidade_diaria[empresa], retorno_medio_diario[empresa],
        s=280 if eh_foco else 150, color=cor_empresa(empresa),
        edgecolor='white', linewidth=1.2, zorder=3 if eh_foco else 2
    )
    ax.annotate(
        empresa, (volatilidade_diaria[empresa], retorno_medio_diario[empresa]),
        textcoords="offset points", xytext=(8, 6),
        fontsize=9.5, fontweight='bold' if eh_foco else 'normal',
        color=cor_empresa(empresa)
    )

# Linhas de referência: mediana do grupo comparável (exclui a NTCO3)
ax.axhline(retorno_medio_diario.drop(EMPRESA_FOCO).median(), color=COR_NEUTRO_REFERENCIA, linewidth=0.8, linestyle=':')
ax.axvline(volatilidade_diaria.drop(EMPRESA_FOCO).median(), color=COR_NEUTRO_REFERENCIA, linewidth=0.8, linestyle=':')

vol_ntco3 = volatilidade_diaria[EMPRESA_FOCO]
ret_ntco3 = retorno_medio_diario[EMPRESA_FOCO]
vol_mediana_comp = volatilidade_diaria.drop(EMPRESA_FOCO).median()
ret_mediana_comp = retorno_medio_diario.drop(EMPRESA_FOCO).median()

if ret_ntco3 >= ret_mediana_comp and vol_ntco3 <= vol_mediana_comp:
    leitura_risco_retorno = f"{EMPRESA_FOCO} combina retorno e risco iguais ou melhores que a mediana do grupo"
elif ret_ntco3 >= ret_mediana_comp and vol_ntco3 > vol_mediana_comp:
    leitura_risco_retorno = f"{EMPRESA_FOCO} tem retorno acima da mediana, mas com volatilidade também acima da mediana do grupo"
elif ret_ntco3 < ret_mediana_comp and vol_ntco3 <= vol_mediana_comp:
    leitura_risco_retorno = f"{EMPRESA_FOCO} tem risco controlado, mas retorno abaixo da mediana do grupo"
else:
    leitura_risco_retorno = f"{EMPRESA_FOCO} tem retorno abaixo da mediana com volatilidade acima da mediana do grupo"

ax.set_title(leitura_risco_retorno)
ax.set_xlabel("Volatilidade diária — desvio padrão dos retornos (%)")
ax.set_ylabel("Retorno médio diário (%)")

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# ETAPA 5.3 — Drawdown (queda em relação ao pico anterior)
# ------------------------------------------------------------------
# Pergunta que este gráfico responde: qual foi a maior queda da NTCO3
# frente ao seu próprio pico no período, e como isso se compara ao
# grupo comparável?

fig, ax = plt.subplots(figsize=(14, 6))

pico_movel = df_pivot.cummax()
drawdown = (df_pivot / pico_movel - 1) * 100  # sempre <= 0

for empresa in empresas_beleza:
    if empresa == EMPRESA_FOCO:
        continue
    ax.plot(drawdown.index, drawdown[empresa], color=cor_empresa(empresa), linewidth=1.0, alpha=0.5, zorder=2)

ax.plot(drawdown.index, drawdown[EMPRESA_FOCO], color=cor_empresa(EMPRESA_FOCO), linewidth=2.5, label=EMPRESA_FOCO, zorder=3)
ax.fill_between(drawdown.index, drawdown[EMPRESA_FOCO], 0, color=cor_empresa(EMPRESA_FOCO), alpha=0.15)

maior_drawdown_por_empresa = drawdown.min()
empresa_maior_drawdown = maior_drawdown_por_empresa.idxmin()
drawdown_ntco3 = maior_drawdown_por_empresa[EMPRESA_FOCO]

if empresa_maior_drawdown == EMPRESA_FOCO:
    titulo_drawdown = f"{EMPRESA_FOCO} teve a maior queda em relação ao próprio pico entre as empresas do grupo"
else:
    titulo_drawdown = f"{EMPRESA_FOCO} teve queda máxima de {formatar_percentual(drawdown_ntco3)}, menor que a de {empresa_maior_drawdown}"

ax.set_title(titulo_drawdown)
ax.set_ylabel("Drawdown em relação ao pico anterior (%)")
ax.set_xlabel("")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# ETAPA 5.4 — Ranking multidimensional
# ------------------------------------------------------------------
# Pergunta que esta tabela responde: indicador a indicador, onde a
# NTCO3 se destaca ou perde frente às comparáveis?
#
# Metodologia: cada indicador é reportado individualmente — não é
# criado nenhum score combinado/arbitrário. A tabela é ordenada por
# Retorno Acumulado (%); os demais indicadores (volatilidade, máximo
# drawdown e correlação com a NTCO3) complementam a leitura sem serem
# somados entre si.

retorno_acumulado_pct = df_kpis['Crescimento Acumulado (%)']
volatilidade_diaria_pct = df_retornos[empresas_beleza].std()
maximo_drawdown_pct = drawdown.min()
correlacao_com_ntco3 = correlacao[EMPRESA_FOCO]

df_ranking = pd.DataFrame({
    'Retorno Acumulado (%)': retorno_acumulado_pct,
    'Volatilidade Diária (%)': volatilidade_diaria_pct,
    'Máximo Drawdown (%)': maximo_drawdown_pct,
    f'Correlação com {EMPRESA_FOCO}': correlacao_com_ntco3,
}).loc[empresas_beleza]

df_ranking = df_ranking.sort_values('Retorno Acumulado (%)', ascending=False)


def destacar_ntco3(linha):
    cor = f"background-color: {cores_marcas[EMPRESA_FOCO]['claro']}" if linha.name == EMPRESA_FOCO else ""
    return [cor] * len(linha)


display(
    df_ranking.style
    .apply(destacar_ntco3, axis=1)
    .format("{:.2f}")
)


In [ ]:
# ==========================================
# CRIANDO A FIGURA DO INFOGRÁFICO
# ==========================================
# ETAPA 4 — Estrutura por blocos narrativos.
# Nesta etapa, os Blocos 1 (Visão Geral) e 3 (Comparação entre
# empresas) são construídos com métricas REAIS, calculadas a partir
# dos dados já carregados. Os Blocos 2 (evolução da NTCO3 no tempo),
# 4 (relações/padrões) e 5 (insights finais) dependem de gráficos que
# ainda não existem (retorno acumulado, série temporal indexada,
# risco x retorno — Etapa 5) e por isso ficam reservados como
# placeholders explícitos, para não inventar números.

fig, ax = plt.subplots(figsize=(12, 18))

ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis('off')

cor_fundo = '#F7F8FA'
cor_branco = '#FFFFFF'
cor_texto = '#263238'
cor_texto_secundario = '#667085'
cor_placeholder = '#E4E7EC'
cor_ntco_escuro = cores_marcas['NTCO3']['escuro']

# ------------------------------------------------------------------
# CÁLCULOS REAIS PARA O BLOCO 1 (Visão Geral)
# ------------------------------------------------------------------
periodo_inicio = df_beleza['DATPRG'].min()
periodo_fim = df_beleza['DATPRG'].max()
texto_periodo = f"{periodo_inicio.strftime('%d/%m/%Y')} a {periodo_fim.strftime('%d/%m/%Y')}"

qtd_empresas = len(empresas_beleza)

ranking_crescimento = df_kpis['Crescimento Acumulado (%)'].sort_values(ascending=False)
empresa_maior_crescimento = ranking_crescimento.index[0]
valor_maior_crescimento = ranking_crescimento.iloc[0]

posicao_ntco3 = list(ranking_crescimento.index).index(EMPRESA_FOCO) + 1
crescimento_ntco3 = ranking_crescimento[EMPRESA_FOCO]

# ------------------------------------------------------------------
# 1. CABEÇALHO
# ------------------------------------------------------------------
ax.add_patch(patches.Rectangle((0, 94), 100, 6, facecolor=cor_ntco_escuro))
ax.text(
    3, 97, f"NTCO3 vs. grupo comparável — {texto_periodo}",
    color=cor_branco, fontsize=13, fontweight='bold', va='center'
)

# ------------------------------------------------------------------
# BLOCO 1 — VISÃO GERAL
# ------------------------------------------------------------------
ax.text(3, 91.5, "BLOCO 1 — VISÃO GERAL", color=cor_texto_secundario, fontsize=11, fontweight='bold')

cartoes_bloco1 = [
    ("Período analisado", texto_periodo, cores_marcas['PNVL3']['claro'], cores_marcas['PNVL3']['escuro']),
    ("Empresas no comparativo", f"{qtd_empresas}", cores_marcas['RADL3']['claro'], cores_marcas['RADL3']['escuro']),
    ("Maior crescimento do grupo", f"{empresa_maior_crescimento}: {formatar_percentual(valor_maior_crescimento)}", cores_marcas['HYPE3']['claro'], cores_marcas['HYPE3']['escuro']),
    ("Posição da NTCO3 (crescimento)", f"{posicao_ntco3}ª de {qtd_empresas} — {formatar_percentual(crescimento_ntco3)}", cor_ntco_escuro, cor_branco),
]

largura_cartao = 23.5
gap = 1.0
x0 = 0
y0_bloco1, altura_bloco1 = 79, 11

for i, (titulo_kpi, valor_kpi, cor_fundo_card, cor_texto_card) in enumerate(cartoes_bloco1):
    x = x0 + i * (largura_cartao + gap)
    ax.add_patch(patches.Rectangle((x, y0_bloco1), largura_cartao, altura_bloco1, facecolor=cor_fundo_card))
    ax.text(x + 1.2, y0_bloco1 + altura_bloco1 - 3, titulo_kpi, color=cor_texto_card, fontsize=8.5, alpha=0.85, wrap=True)
    ax.text(x + 1.2, y0_bloco1 + 2.5, valor_kpi, color=cor_texto_card, fontsize=10.5, fontweight='bold', wrap=True)

# ------------------------------------------------------------------
# BLOCO 3 — COMPARAÇÃO ENTRE EMPRESAS
# ------------------------------------------------------------------
y0_bloco3_label = 74.5
ax.text(3, y0_bloco3_label, "BLOCO 3 — COMPARAÇÃO ENTRE EMPRESAS", color=cor_texto_secundario, fontsize=11, fontweight='bold')

y0_bloco3, altura_bloco3 = 50, 22

# Card A: ranking de crescimento acumulado (todas as empresas, NTCO3 destacada)
ax.add_patch(patches.Rectangle((0, y0_bloco3), 48, altura_bloco3, facecolor=cor_branco, edgecolor=cor_placeholder, linewidth=1))
ax.text(2, y0_bloco3 + altura_bloco3 - 2.5, "Ranking — Crescimento Acumulado (%)", color=cor_texto, fontsize=10, fontweight='bold')

for i, (empresa, valor) in enumerate(ranking_crescimento.items()):
    y_linha = y0_bloco3 + altura_bloco3 - 5.5 - i * 3.0
    eh_foco = empresa == EMPRESA_FOCO
    cor_linha = cor_ntco_escuro if eh_foco else cor_texto
    peso = 'bold' if eh_foco else 'normal'
    ax.text(3, y_linha, f"{i+1}º  {empresa}", color=cor_linha, fontsize=9.5, fontweight=peso)
    ax.text(43, y_linha, formatar_percentual(valor), color=cor_linha, fontsize=9.5, fontweight=peso, ha='right')

# Card B: Teste A/B (NTCO3 vs RADL3) + empresa mais correlacionada
ax.add_patch(patches.Rectangle((50, y0_bloco3), 50, altura_bloco3, facecolor=cor_ntco_escuro))
ax.text(52, y0_bloco3 + altura_bloco3 - 2.5, f"Teste A/B — {EMPRESA_FOCO} vs. RADL3", color=cor_branco, fontsize=10, fontweight='bold')

_conclusao_curta = (
    "sem diferença estatística significativa"
    if p_value >= 0.05 else
    "com diferença estatística significativa"
)
ax.text(
    52, y0_bloco3 + altura_bloco3 - 7,
    f"p-valor = {p_value:.4f} ({_conclusao_curta})",
    color=cor_branco, fontsize=9.5
)

ax.text(
    52, y0_bloco3 + altura_bloco3 - 12,
    f"Empresa com retornos mais correlacionados\naos da {EMPRESA_FOCO}: {empresa_mais_correlacionada}",
    color=cor_branco, fontsize=9.5
)

# ------------------------------------------------------------------
# BLOCOS 2, 4 e 5 — RESERVADOS PARA A ETAPA 5/6
# ------------------------------------------------------------------
# ------------------------------------------------------------------
# BLOCO 2 — DESEMPENHO DA NTCO3 AO LONGO DO TEMPO (dados da Etapa 5)
# ------------------------------------------------------------------
y0_bloco2_label = 46.5
ax.text(3, y0_bloco2_label, "BLOCO 2 — DESEMPENHO DA NTCO3 AO LONGO DO TEMPO", color=cor_texto_secundario, fontsize=11, fontweight='bold')

y0_bloco2, altura_bloco2 = 32, 12

ax.add_patch(patches.Rectangle((0, y0_bloco2), 48, altura_bloco2, facecolor=cor_ntco_escuro))
ax.text(2, y0_bloco2 + altura_bloco2 - 2.8, "Evolução indexada (base 100)", color=cor_branco, fontsize=9.5, fontweight='bold')
ax.text(
    2, y0_bloco2 + 2,
    f"{EMPRESA_FOCO}: {formatar_percentual(variacao_ntco3)} no período\n"
    f"Mediana do grupo: {formatar_percentual(variacao_mediana_grupo)}",
    color=cor_branco, fontsize=9
)

ax.add_patch(patches.Rectangle((50, y0_bloco2), 50, altura_bloco2, facecolor=cor_branco, edgecolor=cor_placeholder, linewidth=1))
ax.text(52, y0_bloco2 + altura_bloco2 - 2.8, "Leitura", color=cor_texto, fontsize=9.5, fontweight='bold')
ax.text(52, y0_bloco2 + 2, titulo_indexado, color=cor_texto, fontsize=9, wrap=True)

# ------------------------------------------------------------------
# BLOCO 4 — RELAÇÕES E PADRÕES (dados da Etapa 5)
# ------------------------------------------------------------------
y0_bloco4_label = 30.5
ax.text(3, y0_bloco4_label, "BLOCO 4 — RELAÇÕES E PADRÕES", color=cor_texto_secundario, fontsize=11, fontweight='bold')

y0_bloco4, altura_bloco4 = 16, 12

ax.add_patch(patches.Rectangle((0, y0_bloco4), 48, altura_bloco4, facecolor=cores_marcas['HYPE3']['claro']))
ax.text(2, y0_bloco4 + altura_bloco4 - 2.8, "Risco × Retorno", color=cores_marcas['HYPE3']['escuro'], fontsize=9.5, fontweight='bold')
ax.text(2, y0_bloco4 + 2, leitura_risco_retorno, color=cores_marcas['HYPE3']['escuro'], fontsize=8.7, wrap=True)

ax.add_patch(patches.Rectangle((50, y0_bloco4), 50, altura_bloco4, facecolor=cores_marcas['RADL3']['claro']))
ax.text(52, y0_bloco4 + altura_bloco4 - 2.8, "Drawdown máximo", color=cores_marcas['RADL3']['escuro'], fontsize=9.5, fontweight='bold')
ax.text(
    52, y0_bloco4 + 2,
    f"{EMPRESA_FOCO}: {formatar_percentual(drawdown_ntco3)}\n"
    f"Maior queda do grupo: {empresa_maior_drawdown}",
    color=cores_marcas['RADL3']['escuro'], fontsize=8.7
)

# ------------------------------------------------------------------
# BLOCO 5 — INSIGHTS FINAIS (reservado para a Etapa 6, storytelling)
# ------------------------------------------------------------------
y_base_bloco5, altura_bloco5 = 2, 12
ax.add_patch(
    patches.Rectangle(
        (0, y_base_bloco5), 100, altura_bloco5,
        facecolor=cor_fundo, edgecolor=cor_placeholder, linewidth=1.2, linestyle='--'
    )
)
ax.text(3, y_base_bloco5 + altura_bloco5 - 3, "BLOCO 5 — Insights finais", color=cor_texto_secundario, fontsize=10, fontweight='bold')
ax.text(
    3, y_base_bloco5 + altura_bloco5 - 7,
    "Ver seção \"Etapa 6 — Storytelling\" ao final deste notebook para os\ninsights completos, com referência ao módulo de sazonalidade da NTCO3.",
    color=cor_texto_secundario, fontsize=8.5, style='italic'
)

plt.tight_layout()
plt.show()


## Módulo de Sazonalidade — NTCO3

Investiga se existem meses ou períodos do ano em que o comportamento da NTCO3 se diferencia dos demais, possivelmente associado a datas sazonais/comerciais (Dia das Mães, Dia dos Namorados, Dia dos Pais, Black Friday, Natal). A sazonalidade só é afirmada se houver evidência de recorrência entre anos — não é assumida de antemão.

In [ ]:
# ==========================================
# MÓDULO DE SAZONALIDADE — NTCO3
# ==========================================
# Preparação dos dados: filtra apenas a NTCO3 e extrai atributos temporais.
# Variável de preço utilizada: PREULT (preço de FECHAMENTO) — mesma
# variável já usada em toda a análise anterior, para manter consistência.
# Variável de volume: QUATOT (quantidade de ações negociadas).

MESES_PT = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

df_ntco3 = df_beleza[df_beleza['CODNEG'] == EMPRESA_FOCO].copy()
df_ntco3['Ano'] = df_ntco3['DATPRG'].dt.year
df_ntco3['NumMes'] = df_ntco3['DATPRG'].dt.month
df_ntco3['NomeMes'] = df_ntco3['NumMes'].apply(lambda m: MESES_PT[m - 1])
df_ntco3['NomeMes'] = pd.Categorical(df_ntco3['NomeMes'], categories=MESES_PT, ordered=True)

# Tratamento de valores ausentes: registros sem preço de fechamento (PREULT)
# ou sem quantidade negociada (QUATOT) não podem entrar na agregação mensal.
_antes = len(df_ntco3)
df_ntco3 = df_ntco3.dropna(subset=['PREULT', 'QUATOT'])
_depois = len(df_ntco3)
if _antes != _depois:
    print(f"Removidos {_antes - _depois} registros da NTCO3 com PREULT ou QUATOT ausentes.")

print(f"Total de pregões da NTCO3 no período: {len(df_ntco3)}")
print(f"Anos presentes no dataset: {sorted(df_ntco3['Ano'].unique())}")
display(df_ntco3[['DATPRG', 'Ano', 'NumMes', 'NomeMes', 'PREULT', 'QUATOT']].head())


In [ ]:
# ------------------------------------------------------------------
# Agregação mensal (Ano-Mês), preservando a sequência cronológica
# ------------------------------------------------------------------
# Preço médio mensal = média do PREULT (preço de FECHAMENTO) no mês.
# Quantidade negociada = soma do QUATOT (ações negociadas) no mês.

df_ntco3['AnoMes'] = df_ntco3['DATPRG'].dt.to_period('M')

df_mensal = df_ntco3.groupby('AnoMes').agg(
    Preco_Medio=('PREULT', 'mean'),
    Quantidade_Negociada=('QUATOT', 'sum'),
    Ano=('Ano', 'first'),
    NumMes=('NumMes', 'first'),
    NomeMes=('NomeMes', 'first'),
).sort_index()

df_mensal['Rotulo'] = df_mensal['NomeMes'].astype(str) + '/' + df_mensal['Ano'].astype(str)

print("Evolução mensal completa (Ano-Mês), em ordem cronológica:")
display(df_mensal[['Rotulo', 'Preco_Medio', 'Quantidade_Negociada']])

# ------------------------------------------------------------------
# Consolidado por mês do ano (todos os anos agregados)
# ------------------------------------------------------------------
df_consolidado_mes = df_mensal.groupby('NomeMes', observed=True).agg(
    Preco_Medio_Historico=('Preco_Medio', 'mean'),
    Quantidade_Media_Historica=('Quantidade_Negociada', 'mean'),
    Numero_de_Anos=('Ano', 'nunique'),
).reindex(MESES_PT).dropna(how='all')

print("\nConsolidado por mês do ano (média entre todos os anos disponíveis):")
display(df_consolidado_mes)


In [ ]:
# ------------------------------------------------------------------
# Gráfico principal: Preço médio mensal × Quantidade negociada
# ------------------------------------------------------------------
# Dois gráficos de linha alinhados verticalmente, com o mesmo eixo X,
# para permitir visualizar se picos de negociação coincidem com
# alterações no preço — sem misturar as duas escalas num único eixo Y.
#
# As marcações de datas comemorativas são HIPÓTESES de investigação
# (faixas discretas de fundo), não explicações causais.

MESES_SAZONAIS = {
    5: ('Dia das Mães', '#F2B6C6'),
    6: ('Dia dos Namorados', '#C6B6F2'),
    8: ('Dia dos Pais', '#B6D4F2'),
    11: ('Black Friday', '#4A4A4A'),
    12: ('Natal', '#2E7D32'),
}

fig, (ax_preco, ax_qtd) = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

x = range(len(df_mensal))
rotulos_x = df_mensal['Rotulo'].tolist()
meses_presentes = set(df_mensal['NumMes'].unique())

for ax in (ax_preco, ax_qtd):
    for i, num_mes in enumerate(df_mensal['NumMes']):
        if num_mes in MESES_SAZONAIS:
            _, cor_evento = MESES_SAZONAIS[num_mes]
            ax.axvspan(i - 0.4, i + 0.4, color=cor_evento, alpha=0.12, zorder=0)

ax_preco.plot(x, df_mensal['Preco_Medio'], color=cor_empresa(EMPRESA_FOCO), linewidth=2.4, marker='o', markersize=4, zorder=3)
ax_preco.set_title(f"{EMPRESA_FOCO}: evolução mensal do preço e volume negociado — há sinais de sazonalidade?")
ax_preco.set_ylabel("Preço médio mensal\nde fechamento (R$)")
ax_preco.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: formatar_moeda(v)))

ax_qtd.plot(x, df_mensal['Quantidade_Negociada'], color=cores_marcas[EMPRESA_FOCO]['principal'], linewidth=2.0, marker='o', markersize=4, zorder=3)
ax_qtd.set_title(
    "Comparação entre preço médio das ações e quantidade negociada ao longo dos meses",
    fontsize=10.5, fontweight='normal', color=COR_TEXTO_SECUNDARIO, loc='left'
)
ax_qtd.set_ylabel("Quantidade negociada\n(ações)")
ax_qtd.set_xticks(list(x))
ax_qtd.set_xticklabels(rotulos_x, rotation=45, ha='right')

# Legenda discreta apenas das datas sazonais que caem dentro do período
handles_sazonais = [
    patches.Patch(color=cor, alpha=0.35, label=nome)
    for mes, (nome, cor) in MESES_SAZONAIS.items() if mes in meses_presentes
]
if handles_sazonais:
    ax_preco.legend(handles=handles_sazonais, loc='upper left', frameon=False, fontsize=8, ncol=min(len(handles_sazonais), 5))

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# Comparação entre anos: mesmo mês em anos diferentes
# ------------------------------------------------------------------
# Objetivo: verificar se um comportamento é recorrente entre anos ou
# apenas um evento isolado de um único ano.

anos_disponiveis = sorted(df_mensal['Ano'].unique())

if len(anos_disponiveis) < 2:
    print(
        f"Atenção: o dataset cobre apenas {len(anos_disponiveis)} ano "
        f"(completo ou parcial). Não é possível comparar o mesmo mês em "
        f"anos diferentes ainda — a evidência de recorrência ficará "
        f"limitada nas conclusões finais deste módulo."
    )
else:
    pivot_preco_ano_mes = df_mensal.pivot_table(
        index='NomeMes', columns='Ano', values='Preco_Medio', observed=True
    ).reindex(MESES_PT).dropna(how='all')
    pivot_qtd_ano_mes = df_mensal.pivot_table(
        index='NomeMes', columns='Ano', values='Quantidade_Negociada', observed=True
    ).reindex(MESES_PT).dropna(how='all')

    cores_anos = [cores_marcas[EMPRESA_FOCO]['claro'], cores_marcas[EMPRESA_FOCO]['escuro']]
    if len(anos_disponiveis) > 2:
        cores_anos = [plt.cm.Oranges(v) for v in np.linspace(0.35, 0.9, len(anos_disponiveis))]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    pivot_preco_ano_mes.plot(kind='bar', ax=axes[0], color=cores_anos[:len(anos_disponiveis)], edgecolor='white')
    axes[0].set_title(f"{EMPRESA_FOCO}: preço médio mensal, ano a ano")
    axes[0].set_ylabel("Preço médio (R$)")
    axes[0].set_xlabel("")
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: formatar_moeda(v)))
    axes[0].legend(title="Ano", frameon=False)
    axes[0].tick_params(axis='x', rotation=0)

    pivot_qtd_ano_mes.plot(kind='bar', ax=axes[1], color=cores_anos[:len(anos_disponiveis)], edgecolor='white')
    axes[1].set_title(f"{EMPRESA_FOCO}: quantidade negociada mensal, ano a ano")
    axes[1].set_ylabel("Quantidade negociada (ações)")
    axes[1].set_xlabel("")
    axes[1].legend(title="Ano", frameon=False)
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.show()


In [ ]:
# ------------------------------------------------------------------
# Sazonalidade consolidada: comportamento médio de cada mês, todos os anos
# ------------------------------------------------------------------
# Responde: quais meses apresentam comportamento historicamente acima
# ou abaixo do padrão (referência = média geral do período)?

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

media_geral_preco = df_consolidado_mes['Preco_Medio_Historico'].mean()
media_geral_qtd = df_consolidado_mes['Quantidade_Media_Historica'].mean()

axes[0].plot(
    df_consolidado_mes.index.astype(str), df_consolidado_mes['Preco_Medio_Historico'],
    color=cor_empresa(EMPRESA_FOCO), marker='o', linewidth=2.2
)
axes[0].axhline(media_geral_preco, color=COR_NEUTRO_REFERENCIA, linestyle='--', linewidth=1, label='Média geral do período')
axes[0].set_ylabel("Preço médio\nhistórico (R$)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, p: formatar_moeda(v)))
axes[0].legend(frameon=False)

mes_maior_preco = df_consolidado_mes['Preco_Medio_Historico'].idxmax()
mes_menor_preco = df_consolidado_mes['Preco_Medio_Historico'].idxmin()
axes[0].set_title(f"Historicamente, {mes_maior_preco} tem o maior preço médio e {mes_menor_preco} o menor")

axes[1].plot(
    df_consolidado_mes.index.astype(str), df_consolidado_mes['Quantidade_Media_Historica'],
    color=cores_marcas[EMPRESA_FOCO]['principal'], marker='o', linewidth=2.0
)
axes[1].axhline(media_geral_qtd, color=COR_NEUTRO_REFERENCIA, linestyle='--', linewidth=1, label='Média geral do período')
axes[1].set_ylabel("Quantidade\nnegociada média")
axes[1].legend(frameon=False)

mes_maior_qtd = df_consolidado_mes['Quantidade_Media_Historica'].idxmax()
mes_menor_qtd = df_consolidado_mes['Quantidade_Media_Historica'].idxmin()
axes[1].set_title(f"Historicamente, {mes_maior_qtd} concentra o maior volume negociado e {mes_menor_qtd} o menor")

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# Picos, anomalias e relação preço × quantidade
# ------------------------------------------------------------------
mes_maior_preco_serie = df_mensal.loc[df_mensal['Preco_Medio'].idxmax(), 'Rotulo']
mes_menor_preco_serie = df_mensal.loc[df_mensal['Preco_Medio'].idxmin(), 'Rotulo']
mes_maior_qtd_serie = df_mensal.loc[df_mensal['Quantidade_Negociada'].idxmax(), 'Rotulo']
mes_menor_qtd_serie = df_mensal.loc[df_mensal['Quantidade_Negociada'].idxmin(), 'Rotulo']

print("=== PICOS E MÍNIMOS (série Ano-Mês) ===")
print(f"Maior preço médio mensal: {mes_maior_preco_serie}")
print(f"Menor preço médio mensal: {mes_menor_preco_serie}")
print(f"Maior quantidade negociada mensal: {mes_maior_qtd_serie}")
print(f"Menor quantidade negociada mensal: {mes_menor_qtd_serie}")

# Variação mês a mês (%)
df_mensal['Var_Preco_%'] = df_mensal['Preco_Medio'].pct_change() * 100
df_mensal['Var_Quantidade_%'] = df_mensal['Quantidade_Negociada'].pct_change() * 100

print("\n=== MAIORES VARIAÇÕES MENSAIS DE PREÇO (top 5 em módulo) ===")
display(
    df_mensal[['Rotulo', 'Var_Preco_%']].dropna()
    .reindex(df_mensal['Var_Preco_%'].abs().sort_values(ascending=False).index)
    .head(5)
)


def classificar_padrao(linha):
    if pd.isna(linha['Var_Preco_%']) or pd.isna(linha['Var_Quantidade_%']):
        return None
    if linha['Var_Preco_%'] > 0 and linha['Var_Quantidade_%'] > 0:
        return 'Aumento simultâneo de preço e quantidade'
    if linha['Var_Quantidade_%'] > 0 and linha['Var_Preco_%'] < 0:
        return 'Quantidade subiu, preço caiu'
    if linha['Var_Preco_%'] > 0 and abs(linha['Var_Quantidade_%']) < 5:
        return 'Preço subiu sem aumento relevante de quantidade'
    return 'Outro padrão'


df_mensal['Padrao'] = df_mensal.apply(classificar_padrao, axis=1)

print("\n=== CONTAGEM DE PADRÕES MENSAIS IDENTIFICADOS ===")
print(df_mensal['Padrao'].value_counts())

# Correlação entre preço médio mensal e quantidade negociada mensal
correlacao_preco_qtd = df_mensal[['Preco_Medio', 'Quantidade_Negociada']].corr().iloc[0, 1]
print("\n=== CORRELAÇÃO (Preço médio mensal x Quantidade negociada mensal) ===")
print(f"Correlação de Pearson: {correlacao_preco_qtd:.4f}")
print("Atenção: correlação não implica causalidade.")


In [ ]:
# ------------------------------------------------------------------
# Insights (derivados diretamente dos cálculos acima)
# ------------------------------------------------------------------
anos_unicos = sorted(df_mensal['Ano'].unique())
qtd_anos = len(anos_unicos)

insights = [
    f"O maior preço médio mensal da {EMPRESA_FOCO} no período foi em {mes_maior_preco_serie}, e o menor em {mes_menor_preco_serie}.",
    f"A maior quantidade negociada mensal ocorreu em {mes_maior_qtd_serie}, e a menor em {mes_menor_qtd_serie}.",
    f"Historicamente (consolidando os {qtd_anos} ano(s) disponíveis), {mes_maior_preco} tem, em média, o maior preço, e {mes_menor_preco} o menor.",
    f"Historicamente, {mes_maior_qtd} concentra, em média, o maior volume negociado, e {mes_menor_qtd} o menor.",
]

if abs(correlacao_preco_qtd) < 0.2:
    leitura_corr = "fraca ou praticamente inexistente"
elif abs(correlacao_preco_qtd) < 0.5:
    leitura_corr = "moderada"
else:
    leitura_corr = "forte"
insights.append(f"A correlação entre preço médio mensal e quantidade negociada mensal é {leitura_corr} ({correlacao_preco_qtd:.2f}).")

for _mes_evento, (_nome_evento, _) in MESES_SAZONAIS.items():
    _nome_mes_pt = MESES_PT[_mes_evento - 1]
    if _mes_evento in meses_presentes and _nome_mes_pt in df_consolidado_mes.index:
        _linha_evento = df_consolidado_mes.loc[_nome_mes_pt]
        _acima_media_preco = _linha_evento['Preco_Medio_Historico'] > media_geral_preco
        _acima_media_qtd = _linha_evento['Quantidade_Media_Historica'] > media_geral_qtd
        insights.append(
            f"No mês de {_nome_evento} ({_nome_mes_pt}), o preço médio fica "
            f"{'acima' if _acima_media_preco else 'abaixo'} da média geral, e a quantidade "
            f"negociada fica {'acima' if _acima_media_qtd else 'abaixo'} da média geral — "
            f"OBSERVAÇÃO coincidente com o período analisado, não uma relação causal comprovada."
        )

print("=== INSIGHTS ===")
for i, texto in enumerate(insights, start=1):
    print(f"{i}. {texto}")

# ------------------------------------------------------------------
# Conclusão sobre sazonalidade
# ------------------------------------------------------------------
print("\n=== CONCLUSÃO ===")
if qtd_anos < 2:
    print(
        f"O dataset cobre apenas {qtd_anos} ano completo/parcial. Não há dados "
        f"suficientes para comparar o comportamento do mesmo mês em anos "
        f"diferentes — portanto, NÃO é possível concluir que existe sazonalidade "
        f"recorrente. O que se observa acima são padrões de UM único período "
        f"(observação), não um padrão comprovadamente repetido (conclusão)."
    )
else:
    _ocorrencias_mes_pico = df_mensal[df_mensal['NomeMes'] == mes_maior_preco]['Ano'].nunique()
    _recorrente = _ocorrencias_mes_pico >= 2 and _ocorrencias_mes_pico == qtd_anos
    if _recorrente:
        print(
            f"O mês de {mes_maior_preco} apresentou o maior preço médio em todos "
            f"os {qtd_anos} anos disponíveis, o que é uma evidência (ainda que "
            f"limitada, dado o número de anos) de possível sazonalidade."
        )
    else:
        print(
            f"Com {qtd_anos} anos de dados, ainda não há evidência consistente de "
            f"recorrência: o mês de destaque não se repete de forma clara entre "
            f"os anos disponíveis. Um número maior de anos seria necessário para "
            f"afirmar sazonalidade com mais confiança."
        )


## Etapa 6 — Storytelling: NTCO3 vs. grupo comparável

> **Nota de transparência:** os números abaixo são os valores REAIS,
> calculados a partir dos seus arquivos `COTAHIST_A2025.TXT` e
> `COTAHIST_A2026.TXT` (rodei a lógica completa do notebook sobre os
> dados reais das 6 empresas para validar e extrair estes resultados).
> Uma ressalva de honestidade: neste ambiente de sandbox não consegui
> rodar a leitura do arquivo bruto inteiro (5,68 milhões de linhas de
> TODAS as ações da B3, ~1,4 GB) por limite de memória (3,9 GB) — então
> executei a mesma função `processar_cotahist` sobre os registros reais
> e completos apenas das 6 empresas deste comparativo (2.354 linhas,
> extraídas dos arquivos originais sem qualquer alteração). A lógica
> testada é exatamente a mesma; recomendo rodar o notebook completo uma
> vez no Google Colab (mais memória disponível) para confirmar a leitura
> do arquivo inteiro.

### 0. Achado crítico encontrado ao validar com dados reais
Dois eventos societários reais, que não apareceriam sem os dados,
foram identificados e corrigidos no código (ver célula de filtro/
definição da NTCO3, logo no início do notebook):
- **NTCO3 → NATU3** (01/07/2025): a Natura&Co Holding foi incorporada
  pela Natura Cosméticos; sem tratar isso, a série da "NTCO3" pararia
  em 01/07/2025 no meio do período. As duas séries foram emendadas.
- **ESPA3: grupamento 10 para 1** (15/06/2026): sem ajuste, a ESPA3
  aparentava um "crescimento" de +831% e volatilidade de 43% — um
  artefato do grupamento, não um resultado real. Após o ajuste
  retroativo, o crescimento acumulado da ESPA3 é -6,85%, com
  volatilidade de 3,07% — em linha com as demais.

### 1. Como a NTCO3 se comporta em relação às demais?
No acumulado do período (índice base 100), a **NTCO3 fica abaixo da
mediana do grupo comparável**: variação de **-38,89%**, contra
**-6,85%** da mediana das comparáveis. No ranking de crescimento
acumulado, a NTCO3 ocupa a **5ª posição entre as 6 empresas** —
só à frente da COTY34 (-66,28%).

### 2. Em quais indicadores a NTCO3 se destaca positivamente?
Isoladamente, a NTCO3 não lidera nenhum dos 4 indicadores do ranking
(Retorno, Volatilidade, Drawdown, Correlação). O único ponto
relativamente favorável: seu drawdown máximo (**-48,53%**) é
menor (melhor) que o da COTY34 (-78,43%) e da ESPA3 (-55,70%) —
embora pior que o de PNVL3, HYPE3 e RADL3.

### 3. Em quais indicadores a NTCO3 apresenta desempenho inferior?
- **Retorno Acumulado:** -38,89% — 2º pior do grupo (só COTY34 é pior).
- **Volatilidade Diária:** 3,22% — 2ª mais alta do grupo (só COTY34 é
  mais volátil), e acima da média das comparáveis (2,65%).

### 4. A NTCO3 apresenta maior ou menor volatilidade?
**Maior.** A NTCO3 apresentou volatilidade diária acima da média do
grupo comparável (3,22% vs. 2,65%).

### 5. Qual empresa apresenta comportamento mais semelhante à NTCO3?
**HYPE3**, com correlação de retornos diários de **0,31** — a mais
alta entre as comparáveis (ainda assim, uma correlação moderada/baixa).

### 6. Qual empresa apresenta comportamento mais diferente?
**COTY34**, com correlação de apenas **0,03** — praticamente nenhuma
relação linear entre os retornos diários da COTY34 e da NTCO3.

### 7. Existe alguma relação relevante entre risco e retorno?
Sim, e desfavorável para a NTCO3: ela **combina retorno abaixo da
mediana com volatilidade acima da mediana** do grupo comparável — o
pior dos quatro quadrantes possíveis no gráfico de dispersão
risco×retorno (Etapa 5.2). Em drawdown, a maior queda da NTCO3
(-48,53%) foi menor que a da COTY34 (-78,43%), mas maior que a de
PNVL3, HYPE3 e RADL3.

### 8. Existem períodos específicos em que a NTCO3 se distancia das demais?
Ver o gráfico de evolução indexada (Etapa 5.1) e o de drawdown (Etapa
5.3) para os trechos exatos — a NTCO3 mantém-se consistentemente
abaixo da mediana do grupo ao longo de boa parte do período, não
apenas num evento pontual.

### 9. Existem padrões ou tendências relevantes? (com sazonalidade)
Do módulo de sazonalidade:
- Maior preço médio mensal (série completa): **Fev/2025**; menor:
  **Jan/2026**.
- Maior quantidade negociada mensal: **Mar/2025**; menor: **Ago/2026**.
- Historicamente (média entre os 2 anos disponíveis), **Fevereiro**
  tem o maior preço médio, e **Dezembro** o menor.
- Correlação entre preço médio mensal e quantidade negociada mensal:
  **0,11** (fraca/praticamente inexistente) — correlação não implica
  causalidade.
- **Evidência de sazonalidade:** o mês de **Fevereiro** apresentou o
  maior preço médio em ambos os anos disponíveis (2025 e 2026) — uma
  evidência de possível sazonalidade, **mas ainda limitada**, já que o
  dataset cobre apenas 2 anos. Não há evidência suficiente, com esses
  2 anos, para afirmar sazonalidade associada às datas comemorativas
  específicas (Dia das Mães, Namorados, Pais, Black Friday, Natal) —
  os meses correspondentes oscilam acima/abaixo da média geral sem um
  padrão consistente e comprovado nos dois anos.

### 10. Principais insights (baseados nos dados reais)
1. A NTCO3 fica na 5ª posição de 6 em crescimento acumulado no
   período (-38,89%, contra mediana de -6,85% do grupo comparável).
2. A NTCO3 combina o pior dos dois mundos em risco-retorno: retorno
   abaixo da mediana e volatilidade acima da mediana do grupo.
3. HYPE3 é a empresa com comportamento mais parecido ao da NTCO3
   (correlação 0,31); COTY34 é a mais diferente (correlação 0,03).
4. Historicamente, Fevereiro é o mês de maior preço médio da NTCO3
   nos 2 anos disponíveis — indício de sazonalidade, ainda que com
   evidência limitada (apenas 2 anos de dados).
5. Dois eventos societários (troca de ticker NTCO3→NATU3 e grupamento
   10:1 da ESPA3) precisaram ser corrigidos manualmente nos dados —
   sem esse tratamento, a comparação entre as 6 empresas estaria
   fundamentalmente distorcida.

### Conclusão geral
No período analisado, a NTCO3 tem desempenho **abaixo** do grupo
comparável de beleza/farma listado na B3: menor retorno acumulado que
a mediana, volatilidade acima da média, e uma combinação
risco-retorno desfavorável. A empresa com comportamento mais parecido
é a HYPE3; a mais descorrelacionada é a COTY34. Há um indício de
sazonalidade em Fevereiro (preço historicamente mais alto), mas com
apenas 2 anos de dados essa leitura ainda é limitada — não há
evidência suficiente para associar com confiança o comportamento da
NTCO3 a datas comemorativas específicas do calendário de varejo.
